In [1]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
df = pd.read_csv("../data/processed/cases.csv")

print(df.shape)
df.head()

(31, 7)


,case_id,nomor_perkara,jenis_perkara,ringkasan_fakta,panjang_teks,text_full,amar_putusan
0,case_001,052/phpu.c.1-ii/2004 demi keadilan berdasarkan...,Pertanahan,p u t u s a n perkara nomor: 052/phpu.c.1-ii/2...,1000,p u t u s a n perkara nomor: 052/phpu.c.1-ii/2...,"mengadili, dan memutus perkara konstitusi pada..."
1,case_002,95/puu-xii/2014 demi keadilan berdasarkan ketu...,Pertanahan,salinan putusan nomor 95/puu-xii/2014 demi kea...,60565,salinan putusan nomor 95/puu-xii/2014 demi kea...,amar putusan mengadili; menyatakan
2,case_003,96/puu-xiv/2016 demi keadilan berdasarkan ketu...,Pertanahan,salinan putusan nomor 96/puu-xiv/2016 demi kea...,91722,salinan putusan nomor 96/puu-xiv/2016 demi kea...,amar putusan sebagai berikut: 1) menyatakan
3,case_004,113-10-18/phpu.dpr-dprd/xvii/2019 demi keadila...,Pertanahan,sal inan ketetapan nomor 113-10-18/phpu.dpr-dp...,912,sal inan ketetapan nomor 113-10-18/phpu.dpr-dp...,mengadili perkara konstitusi pada tingkat pert...
4,case_005,99/puu-xx/2022 demi keadilan berdasarkan ketuh...,Pertanahan,1 s ali n an ketetapan nomor 99/puu-xx/2022 de...,869,1 s ali n an ketetapan nomor 99/puu-xx/2022 de...,memutuskan mencabut permohonan a quo; d. bahwa...


In [3]:
print(df.columns)

Index(['case_id', 'nomor_perkara', 'jenis_perkara', 'ringkasan_fakta',
       'panjang_teks', 'text_full', 'amar_putusan'],
      dtype='object')


In [4]:
documents = df["text_full"].fillna("").astype(str)

In [5]:
vectorizer = TfidfVectorizer(
    max_features=5000,
    stop_words=None
)

tfidf_matrix = vectorizer.fit_transform(documents)

print(tfidf_matrix.shape)

(31, 5000)


In [6]:
def retrieve(query, k=5):

    # ubah query ke vector
    query_vec = vectorizer.transform([query])

    # hitung similarity
    sim = cosine_similarity(query_vec, tfidf_matrix).flatten()

    # ambil top-k index
    top_idx = sim.argsort()[-k:][::-1]

    # ambil hasil
    results = []

    for i in top_idx:

        results.append({
            "case_id": df.iloc[i]["case_id"],
            "nomor_perkara": df.iloc[i]["nomor_perkara"],
            "jenis_perkara": df.iloc[i]["jenis_perkara"],
            "similarity": float(sim[i])
        })

    return pd.DataFrame(results)

In [7]:
retrieve("sengketa tanah tanpa izin", k=5)

,case_id,nomor_perkara,jenis_perkara,similarity
0,case_003,96/puu-xiv/2016 demi keadilan berdasarkan ketu...,Pertanahan,0.195781
1,case_016,50/puu-x/2012 demi keadilan berdasarkan ketuha...,Pertanahan,0.161672
2,case_013,147/puu-xxii/2024 demi keadilan berdasarkan ke...,Pertanahan,0.118814
3,case_011,102/puu-xxii/2024 demi keadilan berdasarkan ke...,Pertanahan,0.100828
4,case_012,116/puu-xxii/2024 demi keadilan berdasarkan ke...,Pertanahan,0.098552


In [8]:
queries = [
    "sengketa tanah tanpa izin",
    "pengujian undang undang kehutanan",
    "perselisihan hasil pemilu presiden"
]

for q in queries:
    print("\nQUERY:", q)
    print(retrieve(q, k=3))


QUERY: sengketa tanah tanpa izin
    case_id                                      nomor_perkara jenis_perkara  \
0  case_003  96/puu-xiv/2016 demi keadilan berdasarkan ketu...    Pertanahan   
1  case_016  50/puu-x/2012 demi keadilan berdasarkan ketuha...    Pertanahan   
2  case_013  147/puu-xxii/2024 demi keadilan berdasarkan ke...    Pertanahan   

   similarity  
0    0.195781  
1    0.161672  
2    0.118814  

QUERY: pengujian undang undang kehutanan
    case_id                                      nomor_perkara jenis_perkara  \
0  case_013  147/puu-xxii/2024 demi keadilan berdasarkan ke...    Pertanahan   
1  case_016  50/puu-x/2012 demi keadilan berdasarkan ketuha...    Pertanahan   
2  case_006  6/puu-xxi/2023 demi keadilan berdasarkan ketuh...    Pertanahan   

   similarity  
0    0.214068  
1    0.203361  
2    0.201181  

QUERY: perselisihan hasil pemilu presiden
    case_id                                      nomor_perkara jenis_perkara  \
0  case_010  1/phpu.pres-xxii/2

In [9]:
import os

os.makedirs("../data/model", exist_ok=True)

import joblib

joblib.dump(vectorizer, "../data/model/vectorizer.pkl")
joblib.dump(tfidf_matrix, "../data/model/tfidf_matrix.pkl")

print("Model tersimpan")

Model tersimpan


In [10]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

df = pd.read_csv("../data/processed/cases.csv")

documents = df["text_full"].fillna("")

vectorizer = TfidfVectorizer(max_features=5000)
tfidf_matrix = vectorizer.fit_transform(documents)

In [11]:
def retrieve(query, k=5):

    query_vec = vectorizer.transform([query])
    sim = cosine_similarity(query_vec, tfidf_matrix).flatten()

    top_idx = sim.argsort()[-k:][::-1]

    results = []

    for i in top_idx:
        results.append({
            "case_id": df.iloc[i]["case_id"],
            "similarity": sim[i]
        })

    return pd.DataFrame(results)

In [12]:
retrieve("tanah sengketa", k=3)

,case_id,similarity
0,case_016,0.248121
1,case_003,0.202404
2,case_011,0.149532
